# Controlled KG Ablation with Llama-3.3-70B-Instruct

The exact same fine-tuned Llama checkpoint is used in both conditions. No additional fine-tuning is performed for the KG-enhanced condition. The only difference is whether KG-derived structured contextual evidence is supplied at inference time.


In [ ]:
%pip install -q \
  torch==2.13.0 \
  transformers==5.16.1 \
  datasets==5.0.1 \
  accelerate==1.14.0 \
  peft==0.20.0 \
  bitsandbytes==0.50.2 \
  requests pandas tqdm sentencepiece


In [ ]:
import json, random, re, string, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from datasets import Dataset, DatasetDict

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_REPO = "https://github.com/Gokcimen/Home_Appliance_Dataset"
!rm -rf /content/Home_Appliance_Dataset
!git clone -q --depth 1 {DATA_REPO}.git /content/Home_Appliance_Dataset
DATA_ROOT = Path("/content/Home_Appliance_Dataset")

def flatten(path):
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    rows = []
    for article in raw["data"]:
        for para in article["paragraphs"]:
            for qa in para["qas"]:
                rows.append({
                    "id":str(qa["id"]),
                    "title":article["title"],
                    "context":para["context"],
                    "question":qa["question"],
                    "answers":{
                        "text":[a["text"] for a in qa["answers"]],
                        "answer_start":[int(a["answer_start"]) for a in qa["answers"]],
                    }
                })
    return rows

raw_datasets = DatasetDict({
    "train":Dataset.from_list(flatten(DATA_ROOT/"train.json")),
    "validation":Dataset.from_list(flatten(DATA_ROOT/"dev.json")),
    "test":Dataset.from_list(flatten(DATA_ROOT/"test.json")),
})

assert len(raw_datasets["train"]) == 8000
assert len(raw_datasets["validation"]) == 1000
assert len(raw_datasets["test"]) == 1000


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
ADAPTER_PATH = "/content/outputs/llama33_70b/final_adapter"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()


In [ ]:
def structured_facts(title, context):
    facts = [f"Product: {title}"]
    sentences = [
        x.strip()
        for x in re.split(r"(?<=[.!?])\s+", context)
        if x.strip()
    ]
    keywords = (
        "capacity","energy","efficiency","noise","dimension",
        "weight","feature","technology","warranty","program",
        "kg","litre","liter","db"
    )
    for sentence in sentences:
        if any(k in sentence.lower() for k in keywords):
            facts.append(sentence)
    return "\n".join(facts)

SYSTEM_PROMPT = (
    "You are a precise question-answering assistant for home-appliance information. "
    "Use the supplied evidence for factual claims. Return only the answer."
)

def build_prompt(example, use_kg):
    kg_evidence = structured_facts(
        example["title"], example["context"]
    ) if use_kg else "None"

    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"Textual context:\n{example['context']}\n\n"
        f"Structured KG evidence:\n{kg_evidence}\n\n"
        f"Question:\n{example['question']}\n\n"
        "Answer:"
    )

def generate_answer(example, use_kg):
    prompt = build_prompt(example, use_kg)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(model.device)

    t0 = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - t0

    prediction = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()
    prediction = prediction.splitlines()[0].strip() if prediction else ""

    return prediction, latency


In [ ]:
def normalize_answer(text):
    text = str(text).lower()
    text = "".join(c for c in text if c not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def exact_match(prediction, reference):
    return float(
        normalize_answer(prediction) == normalize_answer(reference)
    )

def token_f1(prediction, reference):
    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()
    if not pred_tokens or not ref_tokens:
        return float(pred_tokens == ref_tokens)
    common = Counter(pred_tokens) & Counter(ref_tokens)
    same = sum(common.values())
    if same == 0:
        return 0.0
    precision = same / len(pred_tokens)
    recall = same / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def score(rows):
    n = len(rows)
    return {
        "n":n,
        "F1":100 * sum(token_f1(r["prediction"], r["gold"]) for r in rows) / n,
        "EM":100 * sum(exact_match(r["prediction"], r["gold"]) for r in rows) / n,
        "mean_latency_seconds":sum(r["latency_seconds"] for r in rows) / n,
    }


In [ ]:
def run_condition(use_kg):
    rows = []
    for example in raw_datasets["test"]:
        prediction, latency = generate_answer(example, use_kg)
        rows.append({
            "id":example["id"],
            "prediction":prediction,
            "gold":example["answers"]["text"][0],
            "latency_seconds":latency,
            "kg_enabled":bool(use_kg),
        })
    return rows

without_kg = run_condition(False)
with_kg = run_condition(True)

without_metrics = score(without_kg)
with_metrics = score(with_kg)

print("Without KG:", without_metrics)
print("With KG:", with_metrics)
print(
    "Absolute improvement:",
    {
        "F1":with_metrics["F1"] - without_metrics["F1"],
        "EM":with_metrics["EM"] - without_metrics["EM"],
    }
)

for name, rows in [
    ("without_kg", without_kg),
    ("with_kg", with_kg),
]:
    with open(
        f"/content/{name}_llama_predictions.jsonl",
        "w",
        encoding="utf-8",
    ) as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
